In [1]:
import requests
import time

import pandas as pd

# Break


In [2]:
import requests
import pandas as pd
import time

pgs_ids = ["PGS000677", "PGS000699", "PGS000688", "PGS000686", "PGS000671", "PGS000672", "PGS000689", "PGS000675", "PGS002807", "PGS001133", "PGS000684", "PGS000305", "PGS000306", "PGS000685", "PGS001228", "PGS000716", "PGS001227", "PGS001101", "PGS001230", "PGS002308", "PGS000709", "PGS000710", "PGS000703", "PGS000900", "PGS000331", "PGS000039", "PGS003356", "PGS000330", "PGS001134", "PGS003725", "PGS003446", "PGS000329"]


rows = []

for pgs_id in pgs_ids:
    try:
        # Step 1: Retrieve PGS Score information
        score_url = f"https://www.pgscatalog.org/rest/score/{pgs_id}"
        score_response = requests.get(score_url)
        score_response.raise_for_status()
        score_data = score_response.json()

        trait = score_data.get("trait_reported", "NA")
        n_variants = score_data.get("variants_number", "NA")
        ftp_url = score_data.get("ftp_scoring_file", "NA")
        publication = score_data.get("publication", {})
        pgp_id = publication.get("id", "NA")
        pmid = publication.get("PMID", "NA")

        # Extract EFO information if available
        trait_efo_list = score_data.get("trait_efo", [])
        if trait_efo_list:
            efo_entry = trait_efo_list[0]  # Use the first EFO entry
            efo_label = efo_entry.get("label", "NA")
            efo_description = efo_entry.get("description", "NA")
            efo_url = efo_entry.get("url", "NA")
        else:
            efo_label = "NA"
            efo_description = "NA"
            efo_url = "NA"

        # Extract sample training data
        samples_training = score_data.get("samples_training", [])
        if samples_training:
            training_sample = samples_training[0]  # Use the first training sample
            sample_number = training_sample.get("sample_number", "NA")
            ancestry_broad = training_sample.get("ancestry_broad", "NA")
        else:
            sample_number = "NA"
            ancestry_broad = "NA"

        # Step 2: Retrieve Performance Data
        perf_url = "https://www.pgscatalog.org/rest/performance/search"
        perf_response = requests.get(perf_url, params={"pgs_id": pgs_id})
        perf_response.raise_for_status()
        perf_data = perf_response.json()
        results = perf_data.get("results", [])

        if results:
            first_ppm = results[0]  # Use the first Performance Metric
            phenotyping_reported = first_ppm.get("phenotyping_reported", "NA")
            matched_ppm_id = first_ppm.get("id", "NA")

            # Extract effect sizes
            effect_sizes = first_ppm["performance_metrics"].get("effect_sizes", [])
            for effect in effect_sizes:
                row = {
                    "pgs_id": pgs_id,
                    "trait_reported": trait,
                    "n_variants": n_variants,
                    "ftp_scoring_file": ftp_url,
                    "pgp_id": pgp_id,
                    "pubmed_id": pmid,
                    "efo_label": efo_label,
                    #"efo_description": efo_description,
                    #"efo_url": efo_url,
                    "matched_ppm": matched_ppm_id,
                    "phenotyping_reported": phenotyping_reported,
                    "sample_number": sample_number,
                    "ancestry_broad": ancestry_broad,
                    "metric_type": "Effect Size",
                    "name_long": effect.get("name_long", "NA"),
                    "name_short": effect.get("name_short", "NA"),
                    "estimate": effect.get("estimate", "NA"),
                    "ci_lower": effect.get("ci_lower", "NA"),
                    "ci_upper": effect.get("ci_upper", "NA")
                }
                rows.append(row)

            # Extract other metrics
            other_metrics = first_ppm["performance_metrics"].get("othermetrics", [])
            for metric in other_metrics:
                row = {
                    "pgs_id": pgs_id,
                    "trait_reported": trait,
                    "n_variants": n_variants,
                    "ftp_scoring_file": ftp_url,
                    "pgp_id": pgp_id,
                    "pubmed_id": pmid,
                    "efo_label": efo_label,
                    #"efo_description": efo_description,
                    #"efo_url": efo_url,
                    "matched_ppm": matched_ppm_id,
                    "phenotyping_reported": phenotyping_reported,
                    "sample_number": sample_number,
                    "ancestry_broad": ancestry_broad,
                    "metric_type": "Other Metric",
                    "name_long": metric.get("name_long", "NA"),
                    "name_short": metric.get("name_short", "NA"),
                    "estimate": metric.get("estimate", "NA"),
                    "ci_lower": metric.get("ci_lower", "NA"),
                    "ci_upper": metric.get("ci_upper", "NA")
                }
                rows.append(row)
        else:
            # No Performance Metrics found
            row = {
                "pgs_id": pgs_id,
                "trait_reported": trait,
                "n_variants": n_variants,
                "ftp_scoring_file": ftp_url,
                "pgp_id": pgp_id,
                "pubmed_id": pmid,
                "efo_label": efo_label,
                #"efo_description": efo_description,
                #"efo_url": efo_url,
                "matched_ppm": "NA",
                "phenotyping_reported": "NA",
                "sample_number": sample_number,
                "ancestry_broad": ancestry_broad,
                "metric_type": "NA",
                "name_long": "NA",
                "name_short": "NA",
                "estimate": "NA",
                "ci_lower": "NA",
                "ci_upper": "NA"
            }
            rows.append(row)

    except requests.RequestException as e:
        print(f"[ERROR] Failed to fetch data for {pgs_id}: {str(e)}")
        continue
    time.sleep(0.5) 

# Convert the list of rows into a pandas DataFrame
df = pd.DataFrame(rows)

# Display the DataFrame
print(df)


       pgs_id                              trait_reported  n_variants  \
0   PGS000677      Cholesterol [mmol/L] (statin adjusted)       17204   
1   PGS000677      Cholesterol [mmol/L] (statin adjusted)       17204   
2   PGS000699                      Triglycerides [mmol/L]       16003   
3   PGS000699                      Triglycerides [mmol/L]       16003   
4   PGS000688  LDL cholesterol [mmol/L] (statin adjusted)       16184   
5   PGS000688  LDL cholesterol [mmol/L] (statin adjusted)       16184   
6   PGS000686                    HDL cholesterol [mmol/L]       25069   
7   PGS000686                    HDL cholesterol [mmol/L]       25069   
8   PGS000671                      Apolipoprotein A [g/L]       19324   
9   PGS000671                      Apolipoprotein A [g/L]       19324   
10  PGS000672    Apolipoprotein B [g/L] (statin adjusted)       18666   
11  PGS000672    Apolipoprotein B [g/L] (statin adjusted)       18666   
12  PGS000689                      Lipoprotein A [n

In [4]:
df = pd.DataFrame(rows)

# Save to TSV file
#df.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_performance_summary.tsv", sep='\t', index=False)

In [6]:
import requests
import pandas as pd

pgs_ids = ["PGS000677", "PGS000699", "PGS000688", "PGS000686", "PGS000671", "PGS000672", "PGS000689", "PGS000675", "PGS002807", "PGS001133", "PGS000684", "PGS000305", "PGS000306", "PGS000685", "PGS001228", "PGS000716", "PGS001227", "PGS001101", "PGS001230", "PGS002308", "PGS000709", "PGS000710", "PGS000703", "PGS000900", "PGS000331", "PGS000039", "PGS003356", "PGS000330", "PGS001134", "PGS003725", "PGS003446", "PGS000329"]


rows = []

for pgs_id in pgs_ids:
    try:
        # Step 1: Retrieve PGS Score info
        score_url = f"https://www.pgscatalog.org/rest/score/{pgs_id}"
        score_response = requests.get(score_url)
        score_response.raise_for_status()
        score_data = score_response.json()

        trait = score_data.get("trait_reported", "NA")
        n_variants = score_data.get("variants_number", "NA")
        ftp_url = score_data.get("ftp_scoring_file", "NA")
        publication = score_data.get("publication", {})
        pgp_id = publication.get("id", "NA")
        pmid = publication.get("PMID", "NA")

        # EFO
        efo_entry = score_data.get("trait_efo", [{}])[0]
        efo_label = efo_entry.get("label", "NA")

        # Step 2: Retrieve Performance data
        perf_url = "https://www.pgscatalog.org/rest/performance/search"
        perf_response = requests.get(perf_url, params={"pgs_id": pgs_id})
        perf_response.raise_for_status()
        perf_data = perf_response.json()
        results = perf_data.get("results", [])

        if results:
            for ppm in results:
                matched_ppm = ppm.get("id", "NA")
                phenotyping_reported = ppm.get("phenotyping_reported", "NA")

                # Sample ancestry (first sample used)
                samples = ppm.get("sampleset", {}).get("samples", [])
                sample_info = samples[0] if samples else {}
                sample_number = sample_info.get("sample_number", "NA")
                ancestry_broad = sample_info.get("ancestry_broad", "NA")

                # Effect sizes
                for effect in ppm["performance_metrics"].get("effect_sizes", []):
                    rows.append({
                        "pgs_id": pgs_id,
                        "trait_reported": trait,
                        "n_variants": n_variants,
                        "ftp_scoring_file": ftp_url,
                        "pgp_id": pgp_id,
                        "pubmed_id": pmid,
                        "efo_label": efo_label,
                        "matched_ppm": matched_ppm,
                        "phenotyping_reported": phenotyping_reported,
                        "sample_number": sample_number,
                        "ancestry_broad": ancestry_broad,
                        "metric_type": "Effect Size",
                        "metric_name": effect.get("name_short", "NA"),
                        "metric_description": effect.get("name_long", "NA"),
                        "estimate": effect.get("estimate", "NA"),
                        "ci_lower": effect.get("ci_lower", "NA"),
                        "ci_upper": effect.get("ci_upper", "NA")
                    })

                # Other metrics
                for metric in ppm["performance_metrics"].get("othermetrics", []):
                    rows.append({
                        "pgs_id": pgs_id,
                        "trait_reported": trait,
                        "n_variants": n_variants,
                        "ftp_scoring_file": ftp_url,
                        "pgp_id": pgp_id,
                        "pubmed_id": pmid,
                        "efo_label": efo_label,
                        "matched_ppm": matched_ppm,
                        "phenotyping_reported": phenotyping_reported,
                        "sample_number": sample_number,
                        "ancestry_broad": ancestry_broad,
                        "metric_type": "Other Metric",
                        "metric_name": metric.get("name_short", "NA"),
                        "metric_description": metric.get("name_long", "NA"),
                        "estimate": metric.get("estimate", "NA"),
                        "ci_lower": metric.get("ci_lower", "NA"),
                        "ci_upper": metric.get("ci_upper", "NA")
                    })
                    
        else:
            # No metrics found
            rows.append({
                "pgs_id": pgs_id,
                "trait_reported": trait,
                "n_variants": n_variants,
                "ftp_scoring_file": ftp_url,
                "pgp_id": pgp_id,
                "pubmed_id": pmid,
                "efo_label": efo_label,
                "matched_ppm": "NA",
                "phenotyping_reported": "NA",
                "sample_number": "NA",
                "ancestry_broad": "NA",
                "metric_type": "NA",
                "metric_name": "NA",
                "metric_description": "NA",
                "estimate": "NA",
                "ci_lower": "NA",
                "ci_upper": "NA"
            })

    except requests.RequestException as e:
        print(f"[ERROR] Failed to fetch data for {pgs_id}: {str(e)}")
        continue
        
    time.sleep(0.75)
    

# Create DataFrame
df = pd.DataFrame(rows)

# Preview
print(df.head(50))

# Save to CSV (optional)
# df.to_csv("pgs_performance_metrics.csv", index=False)
df.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_performance_summary.tsv", sep='\t', index=False)

       pgs_id                          trait_reported  n_variants  \
0   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
1   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
2   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
3   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
4   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
5   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
6   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
7   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
8   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
9   PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
10  PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
11  PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
12  PGS000677  Cholesterol [mmol/L] (statin adjusted)       17204   
13  PGS000677  Cholesterol [mmol/L

# AUC pulling

In [22]:
import requests
import pandas as pd
import time


pgs_ids = ["PGS000677", "PGS000699", "PGS000688", "PGS000686", "PGS000671", "PGS000672", "PGS000689", "PGS000675", "PGS002807", "PGS001133", "PGS000684", "PGS000305", "PGS000306", "PGS000685", "PGS001228", "PGS000716", "PGS001227", "PGS001101", "PGS001230", "PGS002308", "PGS000709", "PGS000710", "PGS000703", "PGS000900", "PGS000331", "PGS000039", "PGS003356", "PGS000330", "PGS001134", "PGS003725", "PGS003446", "PGS000329"]

rows = []

for pgs_id in pgs_ids:
    try:
        # Step 1: Retrieve PGS Score info
        score_url = f"https://www.pgscatalog.org/rest/score/{pgs_id}"
        score_response = requests.get(score_url)
        score_response.raise_for_status()
        score_data = score_response.json()
        print(pgs_id)

        trait = score_data.get("trait_reported", "NA")
        n_variants = score_data.get("variants_number", "NA")
        ftp_url = score_data.get("ftp_scoring_file", "NA")
        publication = score_data.get("publication", {})
        pgp_id = publication.get("id", "NA")
        pmid = publication.get("PMID", "NA")

        # EFO
        efo_entry = score_data.get("trait_efo", [{}])[0]
        efo_label = efo_entry.get("label", "NA")

        # Step 2: Retrieve Performance data
        perf_url = "https://www.pgscatalog.org/rest/performance/search"
        perf_response = requests.get(perf_url, params={"pgs_id": pgs_id})
        perf_response.raise_for_status()
        perf_data = perf_response.json()
        results = perf_data.get("results", [])

        ancestry_seen = set()

        if results:
            for ppm in results:
                samples = ppm.get("sampleset", {}).get("samples", [])
                if not samples:
                    continue

                sample_info = samples[0]
                ancestry_broad = sample_info.get("ancestry_broad", "NA")

                # Skip if we already collected an entry for this ancestry
                if ancestry_broad in ancestry_seen:
                    continue
                ancestry_seen.add(ancestry_broad)

                matched_ppm = ppm.get("id", "NA")
                phenotyping_reported = ppm.get("phenotyping_reported", "NA")
                sample_number = sample_info.get("sample_number", "NA")

                class_acc = ppm.get("performance_metrics", {}).get("class_acc", [])
                if class_acc:
                    class_metric = class_acc[0]  # Only take the first class_acc metric
                    rows.append({
                        "pgs_id": pgs_id,
                        "trait_reported": trait,
                        "n_variants": n_variants,
                        "ftp_scoring_file": ftp_url,
                        "pgp_id": pgp_id,
                        "pubmed_id": pmid,
                        "efo_label": efo_label,
                        "matched_ppm": matched_ppm,
                        "phenotyping_reported": phenotyping_reported,
                        "sample_number": sample_number,
                        "ancestry_broad": ancestry_broad,
                        "metric_type": "Class Accuracy",
                        "metric_name": class_metric.get("name_short", "NA"),
                        "metric_description": class_metric.get("name_long", "NA"),
                        "estimate": class_metric.get("estimate", "NA"),
                        "ci_lower": class_metric.get("ci_lower", "NA"),
                        "ci_upper": class_metric.get("ci_upper", "NA")
                    })

        # If no relevant entries found at all, add a placeholder row
        if not ancestry_seen:
            rows.append({
                "pgs_id": pgs_id,
                "trait_reported": trait,
                "n_variants": n_variants,
                "ftp_scoring_file": ftp_url,
                "pgp_id": pgp_id,
                "pubmed_id": pmid,
                "efo_label": efo_label,
                "matched_ppm": "NA",
                "phenotyping_reported": "NA",
                "sample_number": "NA",
                "ancestry_broad": "NA",
                "metric_type": "Class Accuracy",
                "metric_name": "NA",
                "metric_description": "NA",
                "estimate": "NA",
                "ci_lower": "NA",
                "ci_upper": "NA"
            })

    except requests.RequestException as e:
        print(f"[ERROR] Failed to fetch data for {pgs_id}: {str(e)}")
        continue

    time.sleep(0.5)

# Create DataFrame and preview
df = pd.DataFrame(rows)
print(df.head(20))

# Optional: Save to file
df.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_class_accuracy_first_per_ancestry.tsv", sep = '\t', index=False)


PGS000677
PGS000699
PGS000688
PGS000686
PGS000671
PGS000672
PGS000689
PGS000675
PGS002807
PGS001133
PGS000684
PGS000305
PGS000306
PGS000685
PGS001228
PGS000716
PGS001227
PGS001101
PGS001230
PGS002308
PGS000709
PGS000710
PGS000703
PGS000900
PGS000331
PGS000039
PGS003356
PGS000330
PGS001134
PGS003725
PGS003446
PGS000329
       pgs_id         trait_reported  n_variants  \
0   PGS000716   Early life body size         295   
1   PGS002308  Type 2 diabetes (T2D)     1259754   
2   PGS002308  Type 2 diabetes (T2D)     1259754   
3   PGS002308  Type 2 diabetes (T2D)     1259754   
4   PGS002308  Type 2 diabetes (T2D)     1259754   
5   PGS000709          Heart failure      183287   
6   PGS000710  Myocardial infarction      183566   
7   PGS000703                 Angina      183692   
8   PGS000039        Ischemic stroke     3225583   
9   PGS000039        Ischemic stroke     3225583   
10  PGS000039        Ischemic stroke     3225583   
11  PGS000039        Ischemic stroke     3225583   
12  

# Pull Training and GWAS DATA 

In [16]:
import requests
import pandas as pd

# List of PGS IDs to fetch
pgs_ids = ["PGS000677", "PGS000699", "PGS000688", "PGS000686", "PGS000671", "PGS000672", "PGS000689", "PGS000675", "PGS002807", "PGS001133", "PGS000684", "PGS000305", "PGS000306", "PGS000685", "PGS001228", "PGS000716", "PGS001227", "PGS001101", "PGS001230", "PGS002308", "PGS000709", "PGS000710", "PGS000703", "PGS000900", "PGS000331", "PGS000039", "PGS003356", "PGS000330", "PGS001134", "PGS003725", "PGS003446", "PGS000329"]
 # Example list

# Store results
pgs_metadata = []
pgs_samples = []
pgs_variants = []

for i, pgs_id in enumerate(pgs_ids, start=1):
    print(f"Processing {i}/{len(pgs_ids)}: {pgs_id}")
    url = f"https://www.pgscatalog.org/rest/score/{pgs_id}"
    response = requests.get(url)
    
    if response.status_code != 200:
        print(f"  ❌ Failed to fetch {pgs_id}")
        continue

    data = response.json()
    
    # General metadata
    metadata = {
        "pgs_id": pgs_id,
        "name": data.get("name"),
        "trait": data.get("trait_reported"),
        "method": data.get("method_name"),
        "variants_number": data.get("variants_number"),
        "date_release": data.get("date_release"),
        "publication_doi": data.get("publication", {}).get("doi"),
    }
    pgs_metadata.append(metadata)

    # Sample training info
    for sample in data.get("samples_training", []):
        sample_info = {
            "pgs_id": pgs_id,
            "sample_number": sample.get("sample_number"),
            "ancestry_broad": sample.get("ancestry_broad"),
            "ancestry_free": sample.get("ancestry_free"),
            "country": sample.get("ancestry_country"),
            "cohorts_additional": sample.get("cohorts_additional"),
            "source_doi": sample.get("source_DOI"),
        }

        # If multiple cohorts exist, concatenate their short names
        cohorts = sample.get("cohorts", [])
        cohort_names = [c.get("name_short") for c in cohorts if c.get("name_short")]
        sample_info["cohort_names"] = ";".join(cohort_names)

        pgs_samples.append(sample_info)
        
    for sample in data.get("samples_variants", []):
        cohort_names = ";".join([
            c.get("name_short") for c in sample.get("cohorts", []) if c.get("name_short")
        ])
        pgs_variants.append({
            "pgs_id": pgs_id,
            "sample_number": sample.get("sample_number"),
            "sample_cases": sample.get("sample_cases"),
            "sample_controls": sample.get("sample_controls"),
            "ancestry_broad": sample.get("ancestry_broad"),
            "ancestry_free": sample.get("ancestry_free"),
            "country": sample.get("ancestry_country"),
            "cohorts_additional": sample.get("cohorts_additional"),
            "cohort_names": cohort_names
        })

# Convert to DataFrames
df_metadata = pd.DataFrame(pgs_metadata)
df_samples = pd.DataFrame(pgs_samples)
df_variants = pd.DataFrame(pgs_variants)

# Show output
print("\nMetadata overview:")
print(df_metadata.head(20))
#df_metadata.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_meta_data.tsv", sep='\t', index=False)

print("\nSample training info:")
print(df_samples.head(20))
#df_samples.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_sample_summary.tsv", sep='\t', index=False)


print("\nVariant Samples:")
print(df_variants.head(20))
df_variants.to_csv("/projects/rpci/songyao/pnfioric/pgs_calc/pgsmix/pgs_paper-revisions/pgs_source_variants_summary.tsv", sep='\t', index=False)


Processing 1/32: PGS000677
Processing 2/32: PGS000699
Processing 3/32: PGS000688
Processing 4/32: PGS000686
Processing 5/32: PGS000671
Processing 6/32: PGS000672
Processing 7/32: PGS000689
Processing 8/32: PGS000675
Processing 9/32: PGS002807
Processing 10/32: PGS001133
Processing 11/32: PGS000684
Processing 12/32: PGS000305
Processing 13/32: PGS000306
Processing 14/32: PGS000685
Processing 15/32: PGS001228
Processing 16/32: PGS000716
Processing 17/32: PGS001227
Processing 18/32: PGS001101
Processing 19/32: PGS001230
Processing 20/32: PGS002308
Processing 21/32: PGS000709
Processing 22/32: PGS000710
Processing 23/32: PGS000703
Processing 24/32: PGS000900
Processing 25/32: PGS000331
Processing 26/32: PGS000039
Processing 27/32: PGS003356
Processing 28/32: PGS000330
Processing 29/32: PGS001134
Processing 30/32: PGS003725
Processing 31/32: PGS003446
Processing 32/32: PGS000329

Metadata overview:
       pgs_id                                name  \
0   PGS000677       snpnet.Cholesterol_a